# Capstone, a read only server

**Scenario:** a maintenance team wants an assistant that reads vibration history for factory assets.
The same database holds the login the control system uses. The server ships with one tool, `run_query`,
marked read only in two places. Both are strings.

Think of **a library reading room**. You may read anything on the shelf, and nothing leaves the
building. That is settled at the door, not printed inside the book.

This capstone builds it properly: resources for the records, one tool for the question people ask, and
a client that finds both.

## Mechanics

A tool goes on the wire with four fields. Two of them do anything.

| Field | Enforced by | What it does |
|---|---|---|
| `name` | The server | Routes the call to your function |
| `description` | Nobody | Text the model reads before it decides. Not a control |
| `inputSchema` | The server, before your code runs | Rejects arguments of the wrong type. It cannot judge a value |
| `annotations.readOnlyHint` | Nobody | A flag for the host interface. The SDK calls these hints and says so in the type itself |

A resource is addressed rather than called, and that is the difference that matters here.

| Field | Meaning |
|---|---|
| `uri` | The address, such as `asset://PM-14`. A template fills one part in per read |
| `mimeType` | How the host should treat the bytes it gets back |
| `resources/read` | The only method there is. Nothing in the protocol writes a resource |

Two rows above are the whole lesson. A description and a hint are read by the model and by a person.
Your code reads neither, so neither stops anything.

## The picture

![The allowlist and the connection mode are the two enforced gates](images/read-only-server.svg)

Every arrow reaching data passes a check your code makes. The credential table has no arrow.

## The cost

```
exposure = tables the connection can reach x statements the tool will accept
```

Both numbers are yours to set. The shipped server leaves them at every table and every statement
SQLite understands.

## The failure

Three tables. Two are the job, and the third is the one nobody meant to ship.

In [1]:
import json
import pathlib
import sqlite3
import tempfile

DB = pathlib.Path(tempfile.mkdtemp()) / "assets.db"
setup = sqlite3.connect(DB)
setup.executescript("""
CREATE TABLE assets(asset_id TEXT PRIMARY KEY, site TEXT, kind TEXT, criticality TEXT);
CREATE TABLE readings(asset_id TEXT, taken_at TEXT, vibration_mm_s REAL);
CREATE TABLE scada_logins(asset_id TEXT, username TEXT, secret TEXT);
INSERT INTO assets VALUES ('PM-14','Duisburg','gearbox','high'),('PM-22','Ghent','pump','low');
INSERT INTO readings VALUES ('PM-14','2026-09-01',7.4),('PM-14','2026-09-02',9.1),('PM-22','2026-09-02',1.2);
INSERT INTO scada_logins VALUES ('PM-14','plc_admin','not-a-real-password');
""")
setup.commit()
setup.close()
print(f"database ready at {DB.name}")

database ready at assets.db


The tool as it shipped: one argument, a plain string, and a promise in the docstring.

In [2]:
import warnings

warnings.filterwarnings("ignore", message="Field 'lifespan'")   # a dependency warns

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

shipped = FastMCP("maintenance", log_level="WARNING")


@shipped.tool(annotations=ToolAnnotations(readOnlyHint=True))
def run_query(sql: str) -> list:
    """Read only. Runs a SELECT against the maintenance database."""
    with sqlite3.connect(DB) as con:
        return con.execute(sql).fetchall()

A host connects and reads the offer. The SDK client is asynchronous and a notebook already owns an
event loop, so the work goes to a worker thread with its own loop and a time limit.

In [3]:
import asyncio
import threading


def run_async(coro, limit=25):
    """Run one coroutine on its own loop, in a thread that cannot outlive us."""
    box = {}

    def worker():
        try:
            box["value"] = asyncio.run(asyncio.wait_for(coro, limit))
        except BaseException as exc:
            box["error"] = exc

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(limit + 10)
    error = box.get("error")
    while isinstance(error, BaseExceptionGroup) and len(error.exceptions) == 1:
        error = error.exceptions[0]        # a group of one is noise, not structure
    if error:
        raise error
    return box["value"]

Now look at what the model is given to decide with.

In [4]:
from mcp.shared.memory import create_connected_server_and_client_session as connect


async def offer(mcp_server):
    """Exactly what a host sees on connecting: the tool list, as sent."""
    async with connect(mcp_server) as session:
        listed = await session.list_tools()
        return [tool.model_dump(exclude_none=True) for tool in listed.tools]


spec = run_async(offer(shipped))[0]
print("description :", spec["description"])
print("annotations :", spec["annotations"])
print("arguments   :", spec["inputSchema"]["properties"])

description : Read only. Runs a SELECT against the maintenance database.
annotations : {'readOnlyHint': True}
arguments   : {'sql': {'title': 'Sql', 'type': 'string'}}


Read only in the text, read only in the annotation. Now send it two statements no reader would need.

In [5]:
count = "SELECT count(*) FROM readings"
start = sqlite3.connect(DB).execute(count).fetchone()[0]

print("credentials:", run_query("SELECT username, secret FROM scada_logins"))
print("delete     :", run_query("DELETE FROM readings WHERE asset_id = 'PM-14'"))

left = sqlite3.connect(DB).execute(count).fetchone()[0]
print(f"readings   : {start} rows before, {left} after")

assert left == start, f"a tool marked read only removed {start - left} rows"

credentials: [('plc_admin', 'not-a-real-password')]
delete     : []
readings   : 3 rows before, 1 after


AssertionError: a tool marked read only removed 2 rows

## The diagnosis

The rows are gone, and a credential came back on the way.

**Nothing in the server read the description.** It is a field on a JSON object, sent to the model so it
can choose. The annotation is the same kind of thing, and the SDK type says so itself.

**The schema did its whole job.** It checked that `sql` is a string, and it is. A schema constrains
shape, never meaning, so a free text argument is a hole with a type on it.

**The connection was the real permission.** It was opened for writing and could see every table. The
blast radius of one tool is every row its connection reaches, and the damage lands before anyone reads
a description.

Read only is a property of the code path, not a claim on the wire.

## The fix

Three layers, each enforced. The first is the driver itself, and it needs one keyword.

In [6]:
import contextlib


def read_only(path):
    """A connection the SQLite driver itself refuses to write through."""
    return sqlite3.connect(f"file:{path}?mode=ro", uri=True)


for sql in ("DELETE FROM readings", "DROP TABLE assets", "SELECT secret FROM scada_logins"):
    with contextlib.closing(read_only(DB)) as con:
        try:
            print(f"{sql:33} {len(con.execute(sql).fetchall())} rows")
        except sqlite3.OperationalError as exc:
            print(f"{sql:33} refused, {exc}")

DELETE FROM readings              refused, attempt to write a readonly database
DROP TABLE assets                 refused, attempt to write a readonly database
SELECT secret FROM scada_logins   1 rows


The driver refuses both writes, and the third line is why one layer is not enough. A read only
connection still reads the credential table. The second layer takes SQL away from the caller: named
queries, bound arguments, and a set of tables the server will name.

In [7]:
QUERIES = {
    "recent_readings": "SELECT taken_at, vibration_mm_s FROM readings "
                       "WHERE asset_id = ? ORDER BY taken_at DESC LIMIT ?",
    "asset_card": "SELECT asset_id, site, kind, criticality FROM assets "
                  "WHERE asset_id = ? LIMIT ?",
}
ALLOWED_TABLES = {"readings", "assets"}

server = FastMCP("maintenance", log_level="WARNING")


@server.tool()
def readings_for(asset_id: str, limit: int = 5) -> list[dict]:
    """Recent vibration readings for one asset, newest first."""
    with contextlib.closing(read_only(DB)) as con:
        cur = con.execute(QUERIES["recent_readings"], (asset_id, min(limit, 50)))
        names = [column[0] for column in cur.description]
        return [dict(zip(names, row)) for row in cur.fetchall()]

`min(limit, 50)` is the third layer, because an argument reaching a query is still one the model chose.
Then the records, which are resources rather than tools because a person picks them by address.

In [8]:
@server.resource("asset://{asset_id}")
def asset_card(asset_id: str) -> str:
    """One asset record, addressed by URI. No method in the protocol writes it."""
    with contextlib.closing(read_only(DB)) as con:
        cur = con.execute(QUERIES["asset_card"], (asset_id, 1))
        names = [column[0] for column in cur.description]
        row = cur.fetchone()
    return json.dumps(dict(zip(names, row)) if row else {})

That is the server. A client connects the way any host would, discovers what is there and uses it,
knowing nothing about SQLite.

In [9]:
async def discover_and_call(mcp_server):
    """Connect, read what is offered, then use one of each kind."""
    async with connect(mcp_server) as session:
        listed = await session.list_tools()
        card = await session.read_resource("asset://PM-22")
        rows = await session.call_tool("readings_for", {"asset_id": "PM-22"})
        return [t.name for t in listed.tools], card.contents[0].text, rows.structuredContent


tools, card, rows = run_async(discover_and_call(server))
print(f"tools offered  : {tools}")
print(f"resource read  : {card}")
print(f"rows returned  : {rows['result']}")
print(f"\nbefore: any statement, every table, writes allowed")
print(f"after : {len(QUERIES)} named reads, {len(ALLOWED_TABLES)} tables, writes refused by the driver")

tools offered  : ['readings_for']
resource read  : {"asset_id": "PM-22", "site": "Ghent", "kind": "pump", "criticality": "low"}
rows returned  : [{'taken_at': '2026-09-02', 'vibration_mm_s': 1.2}]

before: any statement, every table, writes allowed
after : 2 named reads, 2 tables, writes refused by the driver


## The gate

The next person to add a query is the risk. This test reads the allowlist itself, so a new entry has to
pass before it ships.

In [10]:
def test_every_query_is_a_read_of_an_allowed_table():
    for name, sql in QUERIES.items():
        assert sql.startswith("SELECT "), f"{name} is not a read"
        table = sql.split(" FROM ")[1].split()[0]
        assert table in ALLOWED_TABLES, f"{name} reads {table}, which is not on the list"
    try:
        read_only(DB).execute("DELETE FROM readings")
    except sqlite3.OperationalError:
        return
    raise AssertionError("a write reached the database")


test_every_query_is_a_read_of_an_allowed_table()
print(f"gate holds: {len(QUERIES)} named reads, inside {sorted(ALLOWED_TABLES)}")

gate holds: 2 named reads, inside ['assets', 'readings']


Add a query naming `scada_logins` and this test fails before anyone runs the server.

### Enterprise exploration

- A maintenance note is free text an engineer typed. Text inside your data that tries to give the model
  new orders is a real attack here. Which layer stops it, and which only makes it harder?
- Two sites share this server and an operator should see one. Where does the site filter live so a new
  query cannot forget it, and what test proves that at scale?
- The credential table shared the database by accident. What is the compliance exposure if the
  assistant had returned it to a customer, and how fast would you find out?

### Key takeaways

- A description and a `readOnlyHint` are read by the model, never by your server.
- A schema checks the shape of an argument. It cannot tell a query from a deletion.
- Read only is the connection mode plus an allowlist of named queries, both in code.
- Resources are addressed and read. There is no protocol method that writes one.